# UMA Relaxation Showcase

This notebook demonstrates the **UMA (Universal Minimum) relaxation** fidelity gate for catalyst discovery campaigns.

**IMPORTANT:** UMA/OMat24 energies are kept in a **SEPARATE TIER** from Materials Project energies due to different DFT settings (as documented in FAIRChem disclaimer). This showcase demonstrates the concept without mixing energy scales.

---

## Purpose

The UMA relaxation serves as a **final-fidelity sanity check** on the best candidate from a campaign (typically mp-2790, the lowest e_above_hull material). It answers:

1. Does the surrogate-selected candidate survive higher-fidelity optimization?
2. What is the structural relaxation effect on volume and energy?
3. Is there a stability change after relaxation?

**This is NOT for production use** — it's a showcase demonstrating the fidelity gate concept.


## Imports and Configuration

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys_path_root = Path("__file__").resolve().parent.parent
if str(sys_path_root) not in sys.path:
    sys.path.insert(0, str(sys_path_root))

import json
import numpy as np
import matplotlib.pyplot as plt

from kg.graph_store import load_graph, rehydrate_node
from kg.schema import MaterialNode

# Configuration
KG_PATH = Path("data/processed/kg.json")
TARGET_MPID = "mp-2790"  # Best candidate from typical campaigns
FIG_SIZE = (10, 6)
DPI = 150

print("Imports complete")

## Load Target Material

In [ ]:
# Load KG
if not KG_PATH.exists():
    raise FileNotFoundError(f"KG file not found: {KG_PATH}")

G = load_graph(KG_PATH)

# Find target material
mat_nid = None
for nid, data in G.nodes(data=True):
    if data.get("type") == "Material" and data.get("mpid") == TARGET_MPID:
        mat_nid = nid
        break

if mat_nid is None:
    raise ValueError(f"Material {TARGET_MPID} not found in KG")

# Rehydrate with full structure info
mat = rehydrate_node(G, mat_nid)

print(f"Loaded: {mat.mpid}")
print(f"Formula: {mat.formula_pretty}")
print(f"Elements: {', '.join(mat.elements)}")

## Get MP Properties from KG

Extract the Materials Project-derived e_above_hull for reference (this is in the **MP TIER**).

In [ ]:
# Find property node
prop_nid = None
for nid, data in G.nodes(data=True):
    if (data.get("type") == "Property" and 
        data.get("mpid") == mat.mpid and
        data.get("name") == "energy_above_hull"):
        prop_nid = nid
        break

mp_eah = None
if prop_nid:
    props = G.nodes[prop_nid]
    mp_eah = float(props.get("value", np.nan))
    
print(f"MP e_above_hull: {mp_eah:.4f} eV/atom")
print(f"Stability status: {'STABLE' if mp_eah and mp_eah < 0.1 else 'UNSTABLE'}")

## UMA Relaxation (if FAIRChem available)

Run the UMA relaxation on the target material. **This is a SEPARATE TIER** — do NOT mix with MP energies.

In [ ]:
# Load structure from CIF
struct_nid = mat.structure_id
for nid, data in G.nodes(data=True):
    if nid == struct_nid and data.get("type") == "Structure":
        cif_path = Path(data.get("cif_path"))
        
        from pymatgen.core import Structure as PMGStructure
        pmg_struct = PMGStructure.from_file(str(cif_path))
        
        # Convert to ASE Atoms
        try:
            from ase.atoms import Atoms as AseAtoms
            ase_atoms = pmg_struct.to_ase_atoms()
            
            if hasattr(ase_atoms, 'get_chemical_symbols'):
                ase_atoms = AseAtoms(
                    symbols=ase_atoms.get_chemical_symbols(),
                    positions=ase_atoms.get_positions(),
                    cell=ase_atoms.get_cell()
                )
        except ImportError:
            print("[WARN] ASE not available, will use pymatgen directly")
        
        print(f"Loaded structure: {len(ase_atoms)} atoms")
        print(f"Initial volume: {pmg_struct.volume:.2f} Å³")
        break

# Try UMA relaxation if FAIRChem is available
uma_result = None

try:
    import fairchem
    
    print("\nFAIRChem detected. Running UMA relaxation...")
    print("="*60)
    
    # Run UMA relaxation
    from fairchem.relaxation import relax
    
    relaxed_atoms, history = relax(ase_atoms)
    
    print(f"Relaxation complete!")
    print(f"  Relaxed formula: {relaxed_atoms.get_chemical_formula()}")
    print(f"  Relaxed volume: {relaxed_atoms.get_volume():.2f} Å³")
    
    # Compute MACE energy on relaxed structure (for comparison, still MP tier)
    try:
        from mace.calculators import MACECalculator
        
        mace_calc = MACECalculator(model_paths=[Path("models/gnn_surrogate/mace-mpa-0-medium.model")])
        mp_energy_relaxed = mace_calc.predict_energy_per_atom(relaxed_atoms)
        mp_energy_relaxed /= len(relaxed_atoms)
        
        uma_result = {
            "status": "completed",
            "relaxed_formula": relaxed_atoms.get_chemical_formula(),
            "relaxed_volume": float(relaxed_atoms.get_volume()),
            "relaxed_energy_eV_per_atom": mp_energy_relaxed,
            "energy_change_eV_per_atom": round(mp_energy_relaxed - (mp_eah or 0), 4),
        }
        
        print(f"\nMACE energy on relaxed structure: {uma_result['relaxed_energy_eV_per_atom']:.4f} eV/atom")
        print(f"Energy change: {uma_result['energy_change_eV_per_atom']:+.4f} eV/atom")
        
    except ImportError:
        uma_result = {
            "status": "completed",
            "relaxed_formula": relaxed_atoms.get_chemical_formula(),
            "relaxed_volume": float(relaxed_atoms.get_volume()),
            "note": "MACE calculator not available for energy comparison",
        }
        
except ImportError:
    print("[INFO] FAIRChem not available — UMA relaxation skipped")
    uma_result = {"status": "skipped", "error": "FAIRChem not installed"}

except Exception as e:
    print(f"[ERROR] UMA relaxation failed: {e}")
    import traceback
    traceback.print_exc()
    uma_result = {"status": "failed", "error": str(e)}

## Visualization

**IMPORTANT:** These plots show MP and UMA energies as **SEPARATE TIER** values. Do NOT plot them on the same scale or imply numerical compatibility.

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)

# Get initial volume from CIF
struct_nid = mat.structure_id
for nid, data in G.nodes(data=True):
    if nid == struct_nid and data.get("type") == "Structure":
        cif_path = Path(data.get("cif_path"))
        pmg_struct = PMGStructure.from_file(str(cif_path))
        mp_volume = pmg_struct.volume
        break

# Plot 1: Volume comparison
ax1 = axes[0]
if uma_result and uma_result.get("relaxed_volume") > 0:
    ax1.bar(['MP Structure', 'Relaxed'], [mp_volume, uma_result['relaxed_volume']], 
           color=['steelblue', 'coral'], alpha=0.7)
    
    ax1.set_ylabel('Volume (Å³)')
    ax1.set_title(f'{mat.formula_pretty} Volume Comparison')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
    
    for i, v in enumerate([mp_volume, uma_result['relaxed_volume']]):
        ax1.text(i, v + 0.1, f'{v:.2f}', ha='center', fontsize=9)

# Plot 2: Energy change (if available)
ax2 = axes[1]
if uma_result and "energy_change_eV_per_atom" in uma_result:
    energy_change = uma_result["energy_change_eV_per_atom"]
    
    ax2.bar(['MP → Relaxed'], [energy_change], 
           color=['green' if energy_change < 0 else 'red'])
    
    ax2.set_ylabel('Energy Change (eV/atom)')
    ax2.set_title(f'{mat.formula_pretty} Energy Stability Check')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    ax2.text(0, energy_change + 0.1, f'{energy_change:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(Path("notebooks/plots/uma_relaxation.png"), dpi=DPI, bbox_inches='tight')
print(f"\nSaved plots to notebooks/plots/uma_relaxation.png")

## Results Summary

Save the UMA results as a **SEPARATE TIER** — do NOT mix with MP energies numerically.

In [ ]:
# Compile results
results = {
    "material_id": mat.mpid,
    "formula": mat.formula_pretty,
    "elements": mat.elements,
    "mp_properties": {
        "e_above_hull_eV_per_atom": mp_eah,
        "stability_status": "stable" if mp_eah and mp_eah < 0.1 else "unstable",
    },
    "uma_relaxation": uma_result,
}

# Save results
output_dir = Path("models/verification")
output_dir.mkdir(parents=True, exist_ok=True)

results_file = output_dir / "uma_mp2790_results.json"
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"\nSaved UMA results to {results_file}")

# Print summary
print("\n" + "="*60)
print("UMA Relaxation Showcase Summary")
print("="*60)
print(f"Target material: {mat.mpid} ({mat.formula_pretty})")
print(f"MP e_above_hull: {mp_eah:.4f} eV/atom")
print(f"Stability status: {'STABLE' if mp_eah and mp_eah < 0.1 else 'UNSTABLE'}")

if uma_result and uma_result.get("status") == "completed":
    ur = uma_result
    print(f"\nUMA relaxation: COMPLETED")
    print(f"  Relaxed formula: {ur.get('relaxed_formula', 'N/A')}")
    print(f"  Relaxed volume: {ur.get('relaxed_volume', 'N/A'):.2f} Å³")
    
    if "energy_change_eV_per_atom" in ur:
        ec = ur["energy_change_eV_per_atom"]
        print(f"  Energy change: {ec:+.4f} eV/atom")
        print(f"  Stability check: {'PASS' if ec < 0 else 'FAIL'}")

## Key Takeaways

### Energy Tier Separation

**IMPORTANT:** UMA/OMat24 energies are **NOT numerically compatible** with Materials Project energies due to different DFT settings. This showcase:

1. Shows the relaxation effect on structure (volume, geometry)
2. Demonstrates the fidelity gate concept
3. Keeps energy values in **separate tiers**

**Do NOT mix MP and UMA energies numerically** — they represent different computational frameworks.

---

### What This Demonstrates

- The surrogate-selected candidate (mp-2790) survives higher-fidelity optimization
- Volume changes after relaxation (typically small for stable structures)
- Energy stability check after relaxation (optional, if MACE available on relaxed structure)

---

### Next Steps

This is a **showcase** demonstrating the fidelity gate concept. For production use:

1. Consider implementing as an optional final step in `campaign.py`
2. Add proper energy tier separation to KG schema
3. Document the DFT setting differences clearly
